# Data Splitting — ABSA Hotel Santika

Notebook ini membagi dataset hasil pelabelan menjadi **train / validation / test**
untuk keperluan fine-tuning **IndoBERT** pada tugas *Aspect-Based Sentiment Analysis* (ABSA).

## Rasio yang dipakai: 80% Train / 10% Validation / 10% Test

### Justifikasi dari literatur

- **80/10/10** adalah rasio yang umum dan direkomendasikan untuk dataset berukuran
  menengah-besar (belasan ribu sampel). Dengan ~14.7k review, 10% test (~1.5k) dan
  10% validation (~1.5k) sudah cukup besar untuk estimasi metrik yang stabil,
  sementara 80% (~11.8k) memaksimalkan data latih untuk model berbasis Transformer.
- Devlin dkk. (2019) pada **BERT** menggunakan validation/development set terpisah
  untuk *hyperparameter tuning* dan test set hanya untuk evaluasi akhir — kita
  mengikuti skema 3-way split yang sama.
- Pontiki dkk. (2014/2016) pada **SemEval ABSA** memisahkan train dan test set,
  dan praktik umum menambahkan validation dari train untuk *early stopping*.
- Koto dkk. (2020) pada **IndoLEM/IndoBERT** mengevaluasi dengan train/dev/test
  terpisah; rasio 80/10/10 konsisten dengan setup benchmark NLP Bahasa Indonesia.
- Untuk data multi-label dengan kelas minoritas (mis. aspek **Harga** yang jarang),
  splitting acak biasa berisiko membuat distribusi antar-split timpang. Sechidis dkk.
  (2011) memperkenalkan **iterative stratification** untuk multi-label yang menjaga
  proporsi tiap label di setiap split — inilah metode yang kita pakai di sini.

> Referensi: Devlin et al. 2019 (BERT); Pontiki et al. 2014/2016 (SemEval ABSA);
> Koto et al. 2020 (IndoLEM); Sechidis et al. 2011 (Stratification of multi-label data).

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except Exception:
    HAS_ITERSTRAT = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('iterative-stratification tersedia:', HAS_ITERSTRAT)

iterative-stratification tersedia: True


In [2]:
# --- Path dataset ---
BASE = r'c:\\Users\\cencen04_\\Downloads\\ABSA Hotel Santika'
LABELED = os.path.join(BASE, 'Data Labeling', 'dataset_absa_labeled.csv')
OUT_DIR = os.path.join(BASE, 'Data Splitting')
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(LABELED, encoding='utf-8-sig', dtype=str).fillna('')
print('Total baris:', len(df))
df.head(3)

Total baris: 14747


,ID_Review,Platform,Nama_Hotel,Review_Date,Text_Review,Kenyamanan,Kebersihan,Pelayanan,Harga,Lokasi,Fasilitas,Makanan,Alasan_Kenyamanan,Alasan_Kebersihan,Alasan_Pelayanan,Alasan_Harga,Alasan_Lokasi,Alasan_Fasilitas,Alasan_Makanan
0,1,Agoda,Hotel Santika Bandung,5/17/2026,saya selalu menginap di santika bila di bandun...,,positif,positif,,,,positif,,mereka menjaga kebersihan yang baik,keramahtamahan yang baik dari staf,,,,makanan sarapan yang enak
1,2,Agoda,Hotel Santika Bandung,5/11/2026,"bisa jalan kaki langsung ke bip, karena lokasi...",positif,,,,positif,negatif,positif,kamarnya sangat luas dan nyaman,,,,lokasinya strategis,parkirnya relatif kecil,sarapannya 7/10
2,3,Agoda,Hotel Santika Bandung,5/5/2026,lokasinya strategis di pusat kota. makanan saa...,negatif,,,,positif,,positif,air conditioner di kamar agak lambat mungkin s...,,,,lokasinya strategis di pusat kota,,makanan saat sarapan enak dan beragam


## Lingkup data yang di-split

Untuk fine-tuning ABSA, kita hanya memakai review yang **memiliki minimal 1 aspek
terlabel**. Review tanpa aspek (1.465 baris) tidak membawa sinyal sentimen-aspek
sehingga dikeluarkan dari train/val/test (boleh disimpan terpisah untuk analisis).

In [3]:
ASPECTS = ['Lokasi', 'Kenyamanan', 'Pelayanan', 'Kebersihan', 'Harga', 'Makanan', 'Fasilitas']

def has_any_aspect(row):
    return any((row[a] or '').strip() != '' for a in ASPECTS)

mask = df.apply(has_any_aspect, axis=1)
df_labeled = df[mask].reset_index(drop=True)
df_noaspect = df[~mask].reset_index(drop=True)

print('Review berlabel (>=1 aspek):', len(df_labeled))
print('Review tanpa aspek          :', len(df_noaspect))

Review berlabel (>=1 aspek): 13282
Review tanpa aspek          : 1465


## Membangun matriks label untuk *stratified split*

Setiap review punya 7 aspek, masing-masing bisa bernilai
`none / positif / negatif / netral`. Untuk menjaga proporsi setiap kombinasi
aspek-sentimen di tiap split, kita ubah menjadi representasi multi-label:
setiap pasangan `(aspek, sentimen)` menjadi satu kolom biner.

In [4]:
SENTIMENTS = ['positif', 'negatif', 'netral']

# Bangun kolom biner: <aspek>_<sentimen>
Y_cols = []
Y = pd.DataFrame(index=df_labeled.index)
for a in ASPECTS:
    col = df_labeled[a].str.strip().str.lower()
    for s in SENTIMENTS:
        name = f'{a}_{s}'
        Y[name] = (col == s).astype(int)
        Y_cols.append(name)

Ymat = Y.values
print('Bentuk matriks label:', Ymat.shape, '(baris x kombinasi aspek-sentimen)')
Y.sum().sort_values(ascending=False).head(10)

Bentuk matriks label: (13282, 21) (baris x kombinasi aspek-sentimen)


Pelayanan_positif     4945
Lokasi_positif        4079
Kenyamanan_positif    3885
Makanan_positif       3773
Kebersihan_positif    3469
Kenyamanan_negatif    1574
Fasilitas_negatif     1505
Fasilitas_positif     1401
Makanan_negatif        890
Pelayanan_negatif      847
dtype: int64

## Eksekusi split 80/10/10

Strategi:
1. Pisahkan **Test = 10%** dari keseluruhan data berlabel.
2. Dari sisa 90%, pisahkan **Validation** sebesar 1/9 (≈10% dari total).
3. Sisanya menjadi **Train = 80%**.

Jika `iterative-stratification` tersedia, kita pakai **MultilabelStratifiedShuffleSplit**
agar proporsi setiap label terjaga. Jika tidak, fallback ke `train_test_split` biasa.

In [5]:
idx = np.arange(len(df_labeled))

if HAS_ITERSTRAT:
    # Step 1: 90% train+val, 10% test
    msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=RANDOM_STATE)
    trainval_idx, test_idx = next(msss1.split(idx.reshape(-1, 1), Ymat))

    # Step 2: dari train+val, ambil val = 1/9 (~10% total)
    Y_tv = Ymat[trainval_idx]
    msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=1/9, random_state=RANDOM_STATE)
    tr_rel, val_rel = next(msss2.split(trainval_idx.reshape(-1, 1), Y_tv))
    train_idx = trainval_idx[tr_rel]
    val_idx = trainval_idx[val_rel]
    method = 'MultilabelStratifiedShuffleSplit (iterative stratification)'
else:
    trainval_idx, test_idx = train_test_split(idx, test_size=0.10, random_state=RANDOM_STATE, shuffle=True)
    train_idx, val_idx = train_test_split(trainval_idx, test_size=1/9, random_state=RANDOM_STATE, shuffle=True)
    method = 'train_test_split acak (fallback)'

train_df = df_labeled.iloc[np.sort(train_idx)].reset_index(drop=True)
val_df = df_labeled.iloc[np.sort(val_idx)].reset_index(drop=True)
test_df = df_labeled.iloc[np.sort(test_idx)].reset_index(drop=True)

print('Metode :', method)
print('Train  :', len(train_df), f'({len(train_df)/len(df_labeled)*100:.1f}%)')
print('Val    :', len(val_df), f'({len(val_df)/len(df_labeled)*100:.1f}%)')
print('Test   :', len(test_df), f'({len(test_df)/len(df_labeled)*100:.1f}%)')
assert len(train_df)+len(val_df)+len(test_df) == len(df_labeled)

Metode : MultilabelStratifiedShuffleSplit (iterative stratification)
Train  : 10706 (80.6%)
Val    : 1267 (9.5%)
Test   : 1309 (9.9%)


## Verifikasi: distribusi label terjaga di tiap split

Kita bandingkan proporsi setiap sentimen per aspek antara train/val/test.
Proporsi yang mirip menandakan split sudah representatif (tidak bias).

In [6]:
def dist_table(frame):
    rows = {}
    n = len(frame)
    for a in ASPECTS:
        col = frame[a].str.strip().str.lower()
        rows[a] = {
            'positif': round((col == 'positif').mean()*100, 1),
            'negatif': round((col == 'negatif').mean()*100, 1),
            'netral':  round((col == 'netral').mean()*100, 1),
            'none':    round(((col == '') | (col == 'none')).mean()*100, 1),
        }
    return pd.DataFrame(rows).T

print('=== TRAIN (% per aspek) ==='); display(dist_table(train_df))
print('=== VAL   (% per aspek) ==='); display(dist_table(val_df))
print('=== TEST  (% per aspek) ==='); display(dist_table(test_df))

=== TRAIN (% per aspek) ===


,positif,negatif,netral,none
Lokasi,30.5,1.9,1.4,66.2
Kenyamanan,29.0,11.8,1.9,57.3
Pelayanan,36.9,6.3,2.7,54.0
Kebersihan,25.9,4.4,0.6,69.1
Harga,4.5,1.8,1.1,92.6
Makanan,28.2,6.7,4.2,60.9
Fasilitas,10.5,11.2,4.2,74.1


=== VAL   (% per aspek) ===


,positif,negatif,netral,none
Lokasi,32.2,2.1,1.5,64.2
Kenyamanan,30.7,12.5,2.1,54.8
Pelayanan,39.1,6.7,2.9,51.3
Kebersihan,27.4,4.7,0.6,67.4
Harga,4.8,1.9,1.1,92.2
Makanan,29.8,7.0,4.5,58.7
Fasilitas,11.0,11.9,4.5,72.5


=== TEST  (% per aspek) ===


,positif,negatif,netral,none
Lokasi,31.2,2.0,1.5,65.4
Kenyamanan,29.7,12.0,2.0,56.3
Pelayanan,37.8,6.5,2.8,52.9
Kebersihan,26.5,4.5,0.5,68.4
Harga,4.7,1.8,1.1,92.4
Makanan,28.9,6.8,4.4,60.0
Fasilitas,10.7,11.5,4.4,73.4


In [7]:
# Pastikan tidak ada kebocoran (overlap) antar split berdasarkan ID_Review
s_tr = set(train_df['ID_Review']); s_va = set(val_df['ID_Review']); s_te = set(test_df['ID_Review'])
print('Overlap train-val :', len(s_tr & s_va))
print('Overlap train-test:', len(s_tr & s_te))
print('Overlap val-test  :', len(s_va & s_te))
print('Total unik gabungan:', len(s_tr | s_va | s_te))

Overlap train-val : 0
Overlap train-test: 0
Overlap val-test  : 0
Total unik gabungan: 13282


## Simpan hasil split

Output disimpan di folder `Data Splitting/`:
- `train.csv`, `validation.csv`, `test.csv` — untuk fine-tuning.
- `no_aspect.csv` — review tanpa aspek (di luar train/val/test).

Kolom `Alasan_*` ikut disertakan untuk audit, tetapi saat training cukup gunakan
`Text_Review` + kolom 7 aspek sebagai target.

In [8]:
train_path = os.path.join(OUT_DIR, 'train.csv')
val_path   = os.path.join(OUT_DIR, 'validation.csv')
test_path  = os.path.join(OUT_DIR, 'test.csv')
noasp_path = os.path.join(OUT_DIR, 'no_aspect.csv')

train_df.to_csv(train_path, index=False, encoding='utf-8-sig')
val_df.to_csv(val_path, index=False, encoding='utf-8-sig')
test_df.to_csv(test_path, index=False, encoding='utf-8-sig')
df_noaspect.to_csv(noasp_path, index=False, encoding='utf-8-sig')

print('Tersimpan:')
print(' -', train_path, '->', len(train_df), 'baris')
print(' -', val_path, '->', len(val_df), 'baris')
print(' -', test_path, '->', len(test_df), 'baris')
print(' -', noasp_path, '->', len(df_noaspect), 'baris')

Tersimpan:


 - c:\\Users\\cencen04_\\Downloads\\ABSA Hotel Santika\Data Splitting\train.csv -> 10706 baris
 - c:\\Users\\cencen04_\\Downloads\\ABSA Hotel Santika\Data Splitting\validation.csv -> 1267 baris
 - c:\\Users\\cencen04_\\Downloads\\ABSA Hotel Santika\Data Splitting\test.csv -> 1309 baris
 - c:\\Users\\cencen04_\\Downloads\\ABSA Hotel Santika\Data Splitting\no_aspect.csv -> 1465 baris


## Ringkasan

| Split | Jumlah | Proporsi | Kegunaan |
|---|---|---|---|
| Train | ~80% | melatih bobot IndoBERT |
| Validation | ~10% | tuning hyperparameter & early stopping |
| Test | ~10% | evaluasi akhir (sekali pakai) |

Split dilakukan dengan **iterative stratification** (seed=42) sehingga reproducible
dan proporsi tiap aspek-sentimen — termasuk kelas minoritas seperti **Harga** —
terjaga konsisten di ketiga subset.